In [36]:
import findspark
findspark.init()

import os
import json
import pymongo
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [38]:
# %%
# -----------------------------
# Connection configuration
# -----------------------------
mysql_args = {
    "host_name": "localhost",
    "port": "3306",
    "db_name": "adventureworks",
    "conn_props": {
        "user": "root",
        "password": "#Hi10172004",
        "driver": "com.mysql.cj.jdbc.Driver"
    }
}

mongodb_args = {
    "db_name": "adventureworks",
    "collection": "dim_customers_vw"
}
base_dir = os.path.join(os.getcwd(), "project_data")
stream_dir = os.path.join(base_dir, "streaming", "sales_orders")
batch_dir = os.path.join(base_dir, "batch")

employee_csv = os.path.join(batch_dir, "dim_employee.csv")

bronze_dir = os.path.join(base_dir, "bronze")
silver_dir = os.path.join(base_dir, "silver")

In [39]:
# %%
def get_mysql_dataframe(spark_session, sql_query: str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"

    return (
        spark_session.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", args['conn_props']['driver'])
        .option("user", args['conn_props']['user'])
        .option("password", args['conn_props']['password'])
        .option("query", sql_query)
        .load()
    )

In [40]:
# %%
# -----------------------------
# Create Spark session
# -----------------------------
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("AdventureWorks Final Project")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [41]:
# %%
# -----------------------------
# Load dimensions
# -----------------------------

# MySQL dimensions
sql_dim_date = "SELECT * FROM adventureworks.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

sql_dim_products = "SELECT * FROM adventureworks.dim_products_vw"
df_dim_products = get_mysql_dataframe(spark, sql_dim_products, **mysql_args)

Py4JJavaError: An error occurred while calling o98.load.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationPro